# ELVIS Tiling


In [1]:
import os

import geopandas as gpd
import numpy as np
from shapely.geometry import box
from shapely.prepared import prep

In [2]:
# # NCI Filepaths
# aus_boundaries = '/g/data/xe2/cb8590/Outlines/AUS_2021_AUST_GDA2020.shp'
# state_boundaries = '/g/data/xe2/cb8590/Outlines/STE_2021_AUST_GDA2020.shp'
# outdir = '/scratch/xe2/cb8590/lidar/polygons/elvis_inputs/'

# Local filepaths
aus_boundaries = 'AUS_2021_AUST_GDA2020.shp'
state_boundaries = 'STE_2021_AUST_GDA2020.shp'
outdir = '/home/christopher-bradley/repos/shelterbelts/tmpdir/elvis_polygons'

In [3]:
state = 'New South Wales'
tile_size = 30_000   # 30km × 30km in EPSG:7855
crs = 7855

## 1. Clip the state outline and build a regular grid

In [4]:
gdf_states = gpd.read_file(state_boundaries)
state_gdf = gdf_states.loc[gdf_states['STE_NAME21'] == state].to_crs(crs)

# Slightly widened NSW bounds chosen to align with the 2 km BARRA grid.
minx, miny, maxx, maxy = -85_000, 5_845_000, 1_250_000, 6_870_000

xs = np.arange(minx, maxx, tile_size)
ys = np.arange(miny, maxy, tile_size)
tiles = [box(x, y, x + tile_size, y + tile_size) for y in ys for x in xs]
grid = gpd.GeoDataFrame(geometry=tiles, crs=state_gdf.crs)

In [5]:
# Keep only tiles that intersect the state polygon.
state_geom = prep(state_gdf.union_all())
grid = grid[grid.geometry.map(state_geom.intersects)].reset_index(drop=True)
print(f'{len(grid)} tiles intersect {state}')

981 tiles intersect New South Wales


## 2. Write one GeoJSON per tile (for ELVIS downloads)

The Elvis interface lets you choose a geojson, so this creates them for you.

In [6]:
%%time
outdir_geojsons = os.path.join(outdir, f'geojsons_{tile_size}')
os.makedirs(outdir_geojsons, exist_ok=True)

filenames = []
for idx, row in grid.iterrows():
    centroid = row.geometry.centroid
    cx, cy = int(centroid.x), int(centroid.y)
    stub = f'tile{idx}_{cx}_{cy}'
    filenames.append(stub)
    gpd.GeoDataFrame([row], crs=grid.crs).to_crs(4326).to_file(
        os.path.join(outdir_geojsons, f'{stub}.geojson'),
        driver='GeoJSON',
    )

grid['filename'] = filenames
grid.to_file(os.path.join(outdir, f'tiles_{tile_size}_{state.replace(" ", "_")}.gpkg'),
             driver='GPKG')

CPU times: user 5.58 s, sys: 1.03 s, total: 6.62 s
Wall time: 15.6 s
